# AMEX Default Prediction — Portfolio Project Report

## Executive summary

This project predicts customer default risk from anonymized monthly account statements. The central modeling challenge is converting a longitudinal table into leakage-safe customer-level features while preserving information about recent behavior.

The final experimental pipeline combines five standard aggregations, targeted temporal features for 25 high-signal variables, and a regularized LightGBM model. Evaluation uses customer-level stratified three-fold cross-validation with zero train/validation customer overlap.

The selected T11 configuration achieves an OOF ROC-AUC of approximately **0.96146** and an official AMEX metric of approximately **0.79195**. Its gain over the untuned 04.5 model is small but consistent. This notebook reads saved artifacts only; it summarizes evidence and does not retrain models.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 30)

PROJECT_ROOT = find_project_root()
PATHS = {
    'monthly': PROJECT_ROOT/'data/train_data_float16_zstd.feather',
    'labels': PROJECT_ROOT/'data/train_labels.csv',
    'aggregated': PROJECT_ROOT/'02_aggregation/train_aggregated.parquet',
    'refined': PROJECT_ROOT/'04_5_temporal_refined/train_aggregated_temporal_refined.feather',
    'original_importance': PROJECT_ROOT/'04_temporal_features/original_variable_importance.csv',
    'refined_importance': PROJECT_ROOT/'04_5_temporal_refined/temporal_refined_feature_importance.csv',
    'model_comparison': PROJECT_ROOT/'06_model_comparison/model_baseline_comparison_metrics.json',
    'baseline_cv': PROJECT_ROOT/'07_baseline_cv/lightgbm_04_5_cv_metrics.json',
    'baseline_oof': PROJECT_ROOT/'07_baseline_cv/lightgbm_04_5_oof_predictions.parquet',
    'tuning_results': PROJECT_ROOT/'08_hyperparameter_tuning/lightgbm_tuning_results.csv',
    'tuning_summary': PROJECT_ROOT/'08_hyperparameter_tuning/lightgbm_tuning_summary.json',
    'tuned_cv': PROJECT_ROOT/'09_tuned_cv/lightgbm_tuned_cv_metrics.json',
    'tuned_folds': PROJECT_ROOT/'09_tuned_cv/lightgbm_tuned_cv_fold_metrics.csv',
    'tuned_oof': PROJECT_ROOT/'09_tuned_cv/lightgbm_tuned_cv_oof_predictions.parquet',
    'tuned_importance': PROJECT_ROOT/'09_tuned_cv/lightgbm_tuned_cv_feature_importance.csv',
}
missing_files = [str(p) for p in PATHS.values() if not p.exists()]
assert not missing_files, f'Missing report dependencies: {missing_files}'
print(f'Project root: {PROJECT_ROOT}')
print(f'Checked {len(PATHS)} report dependencies: all are present.')


Project root: /Users/amberjiang/Desktop/AMEX
Checked 15 report dependencies: all are present.


## 1. Problem definition: rank risk, not only classify at 0.5

`target=1` denotes default and `target=0` denotes no default. The model produces a continuous risk score, and the competition rewards placing actual defaults near the top of that ranking.

- **ROC-AUC** measures how often a randomly selected default receives a higher score than a randomly selected non-default.
- **Official AMEX metric** averages normalized weighted Gini with top-4% default capture, giving extra importance to the highest-risk segment.
- **Precision, recall, and F1** remain relevant for an operational decision, but they depend on a threshold, review capacity, and error costs that are not specified here.

The project therefore selects models on threshold-independent ranking metrics rather than accuracy.


In [ ]:
def feather_shape(path):
    with pa.memory_map(str(path), 'r') as source:
        reader = ipc.open_file(source)
        return sum(reader.get_batch(i).num_rows for i in range(reader.num_record_batches)), len(reader.schema)

monthly_rows, monthly_cols = feather_shape(PATHS['monthly'])
refined_rows, refined_cols = feather_shape(PATHS['refined'])
agg_meta = pq.ParquetFile(PATHS['aggregated']).metadata
labels = pd.read_csv(PATHS['labels'], usecols=['customer_ID', 'target'])

scale = pd.DataFrame({
    'dataset': ['Monthly records', 'Customers / labels', 'Baseline aggregated', 'Temporal refined'],
    'rows': [monthly_rows, len(labels), agg_meta.num_rows, refined_rows],
    'columns': [monthly_cols, labels.shape[1], agg_meta.num_columns, refined_cols],
})
display(scale.style.format({'rows':'{:,}', 'columns':'{:,}'}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=scale, y='dataset', x='rows', ax=axes[0], color='#4C78A8')
axes[0].set_title('Rows at each data stage'); axes[0].set_xlabel('Rows'); axes[0].set_ylabel('')
axes[0].ticklabel_format(style='plain', axis='x')
for c in axes[0].containers: axes[0].bar_label(c, labels=[f'{v/1e6:.2f}M' for v in scale['rows']], padding=3)
sns.barplot(data=scale, y='dataset', x='columns', ax=axes[1], color='#F58518')
axes[1].set_title('Columns at each data stage'); axes[1].set_xlabel('Columns'); axes[1].set_ylabel('')
for c in axes[1].containers: axes[1].bar_label(c, labels=[f'{v:,}' for v in scale['columns']], padding=3)
plt.tight_layout(); plt.show()

target_counts = labels['target'].value_counts().sort_index()
ax = sns.barplot(x=['Non-default (0)', 'Default (1)'], y=target_counts.values, palette=['#4C78A8','#E45756'])
ax.set_title('Customer-level target distribution')
ax.set_ylabel('Customers'); ax.set_xlabel('')
for i, v in enumerate(target_counts.values): ax.text(i, v, f'{v:,}\n({v/len(labels):.1%})', ha='center', va='bottom')
plt.show()


,dataset,rows,columns
0,Monthly records,"5,531,451",190
1,Customers / labels,"458,913",2
2,Baseline aggregated,"458,913",899
3,Temporal refined,"458,913","1,024"


## 2. Data engineering: monthly statements to customer rows

The raw data contains multiple dated statements per customer, while the target exists once per customer. Splitting monthly rows would leak customer-specific patterns across train and validation sets.

The pipeline first compresses dtypes for laptop-scale processing, then creates a deterministic customer table. Continuous features receive `mean`, `std`, `min`, `max`, and chronological `last`; 11 categorical variables retain their latest state; the final statement date is preserved.

Approximately 5.53 million statement rows become 458,913 unique customer rows before any model split.


## 3. Feature engineering: represent direction and recency

Static aggregates describe level and dispersion but may hide trajectory. Two customers can have the same historical mean while one is improving and the other is deteriorating.

Baseline gain is aggregated to source-variable level, and the top 25 variables receive recent-window and change features. The refined 04.5 dataset replaces raw first values with `last_minus_recent3_mean`, a direct measure of whether the current observation departs from the recent local baseline.

All temporal features use only statements available for that customer and never use the target.


In [ ]:
original_imp = pd.read_csv(PATHS['original_importance']).head(25).sort_values('total_gain')
plt.figure(figsize=(10, 8))
sns.barplot(data=original_imp, y='original_variable', x='gain_share', hue='family', dodge=False)
plt.title('Top 25 original variables by aggregated baseline gain')
plt.xlabel('Share of total gain'); plt.ylabel('Original variable')
plt.legend(title='Family', loc='lower right'); plt.tight_layout(); plt.show()

ref_imp = pd.read_csv(PATHS['refined_importance'])
suffixes = ['last_minus_recent3_mean','last_minus_first','last_minus_mean','recent_3_mean','recent_6_mean']
def temporal_type(name):
    return next((s for s in suffixes if name.endswith('_'+s)), 'baseline aggregation')
ref_imp['feature_type'] = ref_imp['feature'].map(temporal_type)
type_summary = (ref_imp[ref_imp['feature_type']!='baseline aggregation']
                .groupby('feature_type').agg(feature_count=('feature','size'), gain_share=('gain_share','sum'), best_rank=('rank','min'))
                .sort_values('gain_share', ascending=False))
display(type_summary.style.format({'gain_share':'{:.3%}'}))
ax = sns.barplot(data=type_summary.reset_index(), y='feature_type', x='gain_share', color='#54A24B')
ax.set_title('Gain contributed by new temporal feature types')
ax.set_xlabel('Total gain share'); ax.set_ylabel('')
plt.tight_layout(); plt.show()


,feature_count,gain_share,best_rank
feature_type,,,
recent_3_mean,25,10.224%,2
recent_6_mean,25,2.261%,12
last_minus_mean,25,2.125%,16
last_minus_first,25,1.759%,13
last_minus_recent3_mean,25,0.826%,53


## 4. Modeling strategy: interpretable benchmark and nonlinear model

Logistic regression provides a scalable linear benchmark with training-only standardization, imputation, and one-hot encoding. LightGBM natively handles missing values and learns nonlinear thresholds and interactions.

Both models are evaluated on identical validation customers. LightGBM's clear advantage supports the conclusion that important predictive structure is nonlinear rather than merely the result of a flexible preprocessing pipeline.


In [ ]:
comparison = json.loads(PATHS['model_comparison'].read_text())
model_compare = pd.DataFrame([
    {'model':'Logistic Regression', **comparison['logistic_regression']},
    {'model':'LightGBM 04.5', **comparison['lightgbm_04_5']},
])
display(model_compare[['model','roc_auc','amex_metric','training_seconds']].style.format({'roc_auc':'{:.6f}','amex_metric':'{:.6f}','training_seconds':'{:.1f}s'}))
plot_df = model_compare.melt(id_vars='model', value_vars=['roc_auc','amex_metric'], var_name='metric', value_name='score')
ax = sns.barplot(data=plot_df, x='metric', y='score', hue='model')
ax.set_ylim(0.74, 0.98); ax.set_title('Linear baseline vs LightGBM on the same holdout')
ax.set_xlabel(''); ax.set_ylabel('Score')
for c in ax.containers: ax.bar_label(c, fmt='%.4f', padding=3)
plt.tight_layout(); plt.show()


,model,roc_auc,amex_metric,training_seconds
0,Logistic Regression,0.953223,0.772905,110.9s
1,LightGBM 04.5,0.962211,0.792664,554.7s


## 5. Reliability through three-fold cross-validation

A single holdout is useful for iteration but sensitive to sample composition. Stratified three-fold CV gives each customer exactly one out-of-fold prediction and preserves the default rate across folds.

Fold-level dispersion shows uncertainty, while concatenated OOF predictions provide the most defensible overall estimate. The same saved folds are reused for all final candidates, enabling paired comparisons.


In [ ]:
baseline_cv = json.loads(PATHS['baseline_cv'].read_text())
tuned_cv = json.loads(PATHS['tuned_cv'].read_text())
base_folds = pd.DataFrame(baseline_cv['folds']).assign(candidate='04.5 original')
tuned_folds = pd.read_csv(PATHS['tuned_folds'])
folds = pd.concat([base_folds, tuned_folds], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.pointplot(data=folds, x='fold', y='roc_auc', hue='candidate', ax=axes[0], markers=['o','s','D'])
axes[0].set_title('ROC-AUC across customer-level folds'); axes[0].set_ylabel('ROC-AUC')
sns.pointplot(data=folds, x='fold', y='amex_metric', hue='candidate', ax=axes[1], markers=['o','s','D'])
axes[1].set_title('AMEX Metric across customer-level folds'); axes[1].set_ylabel('AMEX Metric')
plt.tight_layout(); plt.show()

overlap_check = folds.groupby('candidate')['customer_overlap'].agg(['max','sum'])
display(overlap_check.rename(columns={'max':'maximum overlap in a fold','sum':'total reported overlap'}))


,maximum overlap in a fold,total reported overlap
candidate,,
04.5 original,0,0
T11,0,0
T8,0,0


## 6. Hyperparameter search: controlled rather than exhaustive

The search varies tree complexity, minimum leaf size, column and row sampling, regularization, and histogram resolution in stages. A fixed holdout screens candidates cheaply; only T11 and T8 advance to full CV.

This approach limits computation and reduces repeated adaptation to the same folds. It also makes each parameter change easier to interpret than a large opaque grid search.


In [ ]:
tuning = pd.read_csv(PATHS['tuning_results'])
completed = tuning[tuning['status']=='completed'].copy()
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(data=completed, x='roc_auc', y='amex_metric', hue='stage', size='training_seconds', sizes=(60, 260))
for _, r in completed.iterrows(): ax.annotate(r['trial_id'].split('_')[0], (r['roc_auc'], r['amex_metric']), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_title('Holdout tuning trials: quality and training cost')
plt.tight_layout(); plt.show()

cv_comparison = pd.DataFrame(tuned_cv['comparison'])
display(cv_comparison[['model','oof_roc_auc','oof_amex_metric','best_iteration_mean','total_training_seconds','delta_oof_auc_vs_original','delta_oof_amex_vs_original']]
        .style.format({'oof_roc_auc':'{:.6f}','oof_amex_metric':'{:.6f}','best_iteration_mean':'{:.1f}','total_training_seconds':'{:.1f}s','delta_oof_auc_vs_original':'{:+.6f}','delta_oof_amex_vs_original':'{:+.6f}'}))


,model,oof_roc_auc,oof_amex_metric,best_iteration_mean,total_training_seconds,delta_oof_auc_vs_original,delta_oof_amex_vs_original
0,04.5 original,0.961343,0.791522,1297.7,1715.6s,+0.000000,+0.000000
1,T11,0.961463,0.791953,813.0,1667.4s,+0.000120,+0.000432
2,T8,0.961433,0.791894,893.7,1599.0s,+0.000091,+0.000373


## 7. What the OOF predictions reveal

The prediction files contain each customer's true target, fold, and out-of-sample risk score. They are not input features; they are the model's auditable answers.

Grouping customers by score decile tests whether observed default rates rise with predicted risk. A strong monotonic lift confirms that the ranking is useful even before a business-specific probability threshold is chosen.


In [ ]:
baseline_oof = pd.read_parquet(PATHS['baseline_oof'])
tuned_oof = pd.read_parquet(PATHS['tuned_oof'])
assert baseline_oof['customer_ID'].is_unique and tuned_oof['customer_ID'].is_unique
assert set(baseline_oof['customer_ID']) == set(tuned_oof['customer_ID'])

roc_inputs = {
    '04.5 original': (baseline_oof['target'], baseline_oof['oof_probability']),
    'T11': (tuned_oof['target'], tuned_oof['T11_probability']),
    'T8': (tuned_oof['target'], tuned_oof['T8_probability']),
}
plt.figure(figsize=(7.5, 6.5))
for name, (y_true, prob) in roc_inputs.items():
    fpr, tpr, _ = roc_curve(y_true, prob)
    auc = roc_auc_score(y_true, prob)
    plt.plot(fpr, tpr, label=f'{name}: {auc:.6f}')
plt.plot([0,1],[0,1],'--',color='gray',label='Random: 0.500000')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('OOF ROC curves'); plt.legend(); plt.tight_layout(); plt.show()

sampled = pd.concat([
    tuned_oof[tuned_oof.target==0].sample(30000, random_state=42),
    tuned_oof[tuned_oof.target==1].sample(30000, random_state=42)
])
sns.histplot(data=sampled, x='T11_probability', hue='target', bins=60, stat='density', common_norm=False, element='step', fill=False)
plt.title('T11 OOF risk-score distribution by actual outcome')
plt.xlabel('Predicted default probability / risk score'); plt.tight_layout(); plt.show()


In [ ]:
risk = tuned_oof[['target','T11_probability']].copy()
risk['risk_decile'] = pd.qcut(risk['T11_probability'], 10, labels=False, duplicates='drop') + 1
deciles = risk.groupby('risk_decile').agg(customers=('target','size'), actual_default_rate=('target','mean'), mean_score=('T11_probability','mean')).reset_index()
ax = sns.barplot(data=deciles, x='risk_decile', y='actual_default_rate', color='#E45756')
ax.set_title('Actual default rate rises sharply in higher T11 risk deciles')
ax.set_xlabel('Predicted risk decile (10 = highest risk)'); ax.set_ylabel('Actual default rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
plt.tight_layout(); plt.show()
display(deciles.style.format({'customers':'{:,}','actual_default_rate':'{:.2%}','mean_score':'{:.4f}'}))


,risk_decile,customers,actual_default_rate,mean_score
0,1,"45,892",0.05%,0.0007
1,2,"45,891",0.11%,0.0013
2,3,"45,891",0.20%,0.0025
3,4,"45,891",0.56%,0.0055
4,5,"45,892",1.43%,0.0144
5,6,"45,891",5.82%,0.0523
6,7,"45,891",22.59%,0.2100
7,8,"45,891",51.98%,0.5272
8,9,"45,891",79.87%,0.8074
9,10,"45,892",96.33%,0.9621


## 8. What the model learned—and what importance cannot prove

Gain importance measures cumulative training-loss reduction from splits on each feature. Averaging it across folds highlights variables that contribute consistently rather than in one fitted booster only.

Importance does not establish causality, effect direction, or individual-customer reasoning. Correlated variables may share or substitute for gain. SHAP analysis would be the appropriate next step for local explanations.


In [ ]:
tuned_imp = pd.read_csv(PATHS['tuned_importance'])
t11_imp = (tuned_imp[tuned_imp.candidate=='T11'].groupby('feature', as_index=False)['gain'].mean())
t11_imp['gain_share'] = t11_imp['gain'] / t11_imp['gain'].sum()
top20 = t11_imp.nlargest(20, 'gain_share').sort_values('gain_share')
plt.figure(figsize=(10, 7.5))
ax = sns.barplot(data=top20, y='feature', x='gain_share', color='#4C78A8')
ax.set_title('T11 top 20 features by mean CV gain')
ax.set_xlabel('Gain share'); ax.set_ylabel('')
plt.tight_layout(); plt.show()
display(top20.sort_values('gain_share', ascending=False).style.format({'gain':'{:,.0f}','gain_share':'{:.2%}'}))


,feature,gain,gain_share
726,P_2_last,"2,071,016",48.58%
733,P_2_recent_3_mean,"302,339",7.09%
55,B_1_last,"226,013",5.30%
222,B_9_last,"56,926",1.34%
115,B_2_last,"55,512",1.30%
808,R_1_recent_3_mean,"52,877",1.24%
801,R_1_last,"42,250",0.99%
432,D_39_last,"34,060",0.80%
5,B_11_last,"33,280",0.78%
457,D_42_mean,"32,089",0.75%


## 9. Leakage and reliability audit

The strongest safeguards in this project are:

1. aggregation creates one row per unique customer before splitting;
2. every CV fold has zero train/validation customer overlap;
3. identifiers and target are excluded from the feature matrix;
4. temporal features use only a customer's observed statements;
5. logistic preprocessing is fit on training customers only;
6. OOF predictions and targets are aligned by customer;
7. reference and tuned models use identical folds.

These checks make the OOF ranking results credible. They do not prove probability calibration or performance under future temporal drift.


## 10. Final selection, limitations, and next steps

### Selected candidate

- Feature set: 04.5 refined temporal features (1,022 model columns)
- Model: T11 LightGBM
- OOF ROC-AUC: approximately **0.961463**
- OOF AMEX metric: approximately **0.791953**
- Improvement over original 04.5 parameters: about `+0.000120` AUC and `+0.000432` AMEX metric

The improvement is stable but modest, suggesting diminishing returns from additional parameter search.

### Highest-value next work

1. add missingness dynamics such as recent missing rate and months since last observed value;
2. create a temporal validation design that better reflects deployment drift;
3. evaluate calibration and thresholds only after business costs are defined;
4. add SHAP for global and customer-level explanations;
5. build the test-data transformation, full-training fit, and submission pipeline.

The current project completes model research and selection, but it does not yet include a test-set submission.
